# Step 4d: Vision Transformer — HSI Glioblastoma Detection

**Architecture:** Custom ViT from scratch (PyTorch) — linear token embedding + learnable positional embeddings + Transformer encoder (pre-LN) + CLS classification head  
**GPU:** A100  
**Grid:** 16 combos (PCA/MI/LASSO x5 + FullSpectrum) x 3 LOPOCV folds = 48 runs  
**Ablation:** patch sizes 1x1, 6x6, 11x11 (separate CSV); patch=1 uses token_size=1 (pixel-level ViT)  

Run cells in order. Safe to interrupt and re-run — completed folds are skipped automatically.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies and clone repo for utils/
import subprocess
subprocess.run(['pip', 'install', '-q', 'h5py', 'scikit-learn', 'tqdm'], check=True)

import os
if not os.path.exists('/content/hsi_project/.git'):
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/Chirag-Mokashi/hsi-cancer-detection.git',
                    '/content/hsi_project'], check=True)
else:
    subprocess.run(['git', '-C', '/content/hsi_project', 'pull', '-q'], check=True)

import sys
sys.path.insert(0, '/content/hsi_project')

from pathlib import Path
# READ: h5 files and band JSONs
DATA_DIR  = Path('/content/drive/MyDrive/HSI')
# WRITABLE: results, checkpoints, plots
OUT_DIR   = Path('/content/drive/MyDrive/HSI')

import utils.data_loader as _dl
_dl.PREPROCESSED_DIR = DATA_DIR / 'preprocessed'
_dl.BAND_SEL_DIR     = DATA_DIR / 'band_selection'

from utils.data_loader import (get_lopocv_folds, load_band_indices,
                                get_experiment_grid, compute_class_weights)
print('Setup complete.')
print('Data  dir:', DATA_DIR)
print('Output dir:', OUT_DIR)

In [ ]:
import csv, math, time
import numpy as np
import h5py
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
print('Imports OK')

In [ ]:
# ============================================================
# CONFIG  — edit VERSION to 'v2' for re-runs / hypertuning
# ============================================================
MODEL   = 'ViT'
VERSION = 'v1'

# Loss function: 'CrossEntropy' (Softmax + CE) or 'BCE' (Sigmoid + BCE)
LOSS_FN = 'CrossEntropy'

# OUT_DIR is set in install-clone cell (writable MyDrive)
RESULTS_DIR  = OUT_DIR / 'results' / 'ViT'
CKPT_DIR     = RESULTS_DIR / 'checkpoints'
PLOTS_DIR    = RESULTS_DIR / 'plots'
for _d in [RESULTS_DIR, CKPT_DIR, PLOTS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

RESULTS_CSV  = RESULTS_DIR / 'vit_{}_results.csv'.format(VERSION)
SUMMARY_CSV  = RESULTS_DIR / 'vit_{}_summary.csv'.format(VERSION)
ABLATION_CSV = RESULTS_DIR / 'vit_{}_ablation.csv'.format(VERSION)

DEVICE          = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RANDOM_SEED     = 42
PATCH_SIZE      = 11     # default for main runs
TOKEN_SIZE      = 4      # spatial size of each token sub-patch
PATCHES_PER_ROI = 300
BATCH_SIZE      = 32
MAX_EPOCHS      = 50
ES_PATIENCE     = 10
LR_INIT         = 1e-4
LR_FACTOR       = 0.5
LR_PATIENCE     = 3
LR_MIN          = 1e-6
WARMUP_EPOCHS   = 5      # linear LR warmup before ReduceLROnPlateau kicks in
VAL_SPLIT       = 0.2
ABLATION_SIZES  = [1, 6, 11]

# ViT architecture hyperparameters
EMBED_DIM  = 64   # token embedding dimension
N_HEADS    = 4    # attention heads (EMBED_DIM must be divisible by N_HEADS)
N_LAYERS   = 4    # transformer encoder blocks
MLP_RATIO  = 4    # MLP hidden dim = MLP_RATIO * EMBED_DIM
DROPOUT    = 0.1

CSV_COLS = ['model', 'method', 'n_bands', 'patch_size', 'token_size', 'fold', 'loss_fn',
            'accuracy', 'sensitivity', 'specificity', 'f1', 'auc',
            'train_time_sec', 'inference_time_per_image_ms']
ABLATION_COLS = CSV_COLS + ['is_ablation']
METRIC_COLS   = ['accuracy', 'sensitivity', 'specificity', 'f1', 'auc',
                 'train_time_sec', 'inference_time_per_image_ms']

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print('Device     :', DEVICE)
print('Results CSV:', RESULTS_CSV)

In [ ]:
# ============================================================
# UTILITY FUNCTIONS
# ============================================================

def is_done(csv_path, model, method, n_bands, patch_size, token_size, fold):
    """Return True if this run's row already exists in the CSV."""
    csv_path = Path(csv_path)
    if not csv_path.exists():
        return False
    with open(csv_path, 'r', newline='') as fh:
        for row in csv.DictReader(fh):
            if (row.get('model')      == str(model)      and
                row.get('method')     == str(method)     and
                row.get('n_bands')    == str(n_bands)    and
                row.get('patch_size') == str(patch_size) and
                row.get('token_size') == str(token_size) and
                row.get('fold')       == str(fold)):
                return True
    return False


def append_row(csv_path, row, cols):
    """Append one row to CSV, writing header if file is new."""
    csv_path = Path(csv_path)
    write_hdr = not csv_path.exists()
    with open(csv_path, 'a', newline='') as fh:
        writer = csv.DictWriter(fh, fieldnames=cols, extrasaction='ignore')
        if write_hdr:
            writer.writeheader()
        writer.writerow(row)


def generate_summary(results_csv, summary_csv, cols, metric_cols):
    """Write mean+-std summary grouped by (method, n_bands, patch_size, token_size)."""
    from collections import defaultdict
    data = defaultdict(lambda: {m: [] for m in metric_cols})
    with open(results_csv, 'r', newline='') as fh:
        for row in csv.DictReader(fh):
            key = (row['method'], row['n_bands'],
                   row.get('patch_size', '11'), row.get('token_size', '4'))
            for m in metric_cols:
                if m in row and row[m] != '':
                    data[key][m].append(float(row[m]))
    sum_cols = ['model', 'method', 'n_bands', 'patch_size', 'token_size',
                'fold', 'loss_fn'] + metric_cols
    with open(summary_csv, 'w', newline='') as fh:
        writer = csv.DictWriter(fh, fieldnames=sum_cols)
        writer.writeheader()
        for (method, n_bands, patch_size, token_size), metrics in sorted(data.items()):
            row = {'model': MODEL, 'method': method, 'n_bands': n_bands,
                   'patch_size': patch_size, 'token_size': token_size,
                   'fold': 'summary', 'loss_fn': LOSS_FN}
            for m, vals in metrics.items():
                arr = np.array(vals)
                row[m] = '{:.4f}+-{:.4f}'.format(arr.mean(), arr.std()) if len(arr) else ''
            writer.writerow(row)
    print('Summary ->', summary_csv)


def save_learning_curve(train_losses, val_losses, title, save_path):
    """Save train/val loss curve to PNG."""
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(train_losses, label='Train loss')
    ax.plot(val_losses,   label='Val loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(save_path, dpi=100)
    plt.close(fig)

print('Utilities defined.')

In [ ]:
# ============================================================
# TOKEN EXTRACTION  (block-based: 1 h5py read per file)
#
# Each patch (patch_size x patch_size) is divided into a grid
# of (token_size x token_size) sub-patches. Each sub-patch is
# flattened to a token of dim = token_size^2 * n_bands.
#
# n_tokens = ceil(patch_size / token_size)^2
# Patch is zero-padded to grid_size*token_size if not divisible.
# ============================================================

def extract_tokens(h5_paths, band_indices, patch_size, token_size, patches_per_roi, seed):
    """
    Block-based token extraction for ViT.

    Returns
    -------
    X : (N, n_tokens, token_dim) float32
    y : (N,) int64
    """
    rng      = np.random.default_rng(seed)
    half     = patch_size // 2
    n_bands  = len(band_indices)
    blk_rows = patch_size + 10

    grid_size = math.ceil(patch_size / token_size)
    padded    = grid_size * token_size
    n_tokens  = grid_size * grid_size
    token_dim = token_size * token_size * n_bands

    all_X, all_y = [], []

    for fpath in h5_paths:
        with h5py.File(fpath, 'r') as f:
            n_rows, n_cols, _ = f['cube'].shape
            label_int = 1 if str(f.attrs['label']) == 'T' else 0

            max_start = n_rows - blk_rows
            start_row = int(rng.integers(0, max(max_start, 1) + 1))
            block = f['cube'][start_row : start_row + blk_rows, :, :]

        block = block[:, :, band_indices].astype(np.float32)  # (blk, cols, n_bands)

        r_min, r_max = half, blk_rows - patch_size + half
        c_min, c_max = half, n_cols   - patch_size + half
        n_valid_r = max(r_max - r_min + 1, 1)
        n_valid_c = max(c_max - c_min + 1, 1)
        n_valid   = n_valid_r * n_valid_c
        n_sample  = min(patches_per_roi, n_valid)

        flat_idx = rng.choice(n_valid, size=n_sample, replace=False)
        rs = r_min + (flat_idx // n_valid_c)
        cs = c_min + (flat_idx %  n_valid_c)

        tokens = np.zeros((n_sample, n_tokens, token_dim), dtype=np.float32)

        for i, (r, c) in enumerate(zip(rs, cs)):
            patch = block[r - half : r - half + patch_size,
                          c - half : c - half + patch_size, :]

            # Pad to grid_size*token_size if patch_size not divisible by token_size
            if padded != patch_size:
                pad_h = padded - patch.shape[0]
                pad_w = padded - patch.shape[1]
                patch = np.pad(patch, ((0, pad_h), (0, pad_w), (0, 0)), mode='reflect')

            tok_idx = 0
            for tr in range(grid_size):
                for tc in range(grid_size):
                    sub = patch[tr * token_size : (tr + 1) * token_size,
                                tc * token_size : (tc + 1) * token_size, :]
                    tokens[i, tok_idx] = sub.reshape(-1)
                    tok_idx += 1

        all_X.append(tokens)
        all_y.append(np.full(n_sample, label_int, dtype=np.int64))

    return np.concatenate(all_X, axis=0), np.concatenate(all_y)


def make_loaders(fold_dict, band_indices, patch_size, token_size,
                 patches_per_roi, seed, batch_size):
    """Extract tokens for a fold and return DataLoaders (train, val, test)."""
    X_all, y_all = extract_tokens(
        fold_dict['train_files'], band_indices, patch_size, token_size,
        patches_per_roi, seed)
    X_test, y_test = extract_tokens(
        fold_dict['test_files'], band_indices, patch_size, token_size,
        patches_per_roi, seed + 1)

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_all, y_all, test_size=VAL_SPLIT, random_state=seed, stratify=y_all)

    def to_loader(X, y, shuffle):
        ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
        # Change num_workers=0 if DataLoader multiprocessing fails on Colab
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                          num_workers=2, pin_memory=True)

    return (to_loader(X_tr, y_tr, True), to_loader(X_val, y_val, False),
            to_loader(X_test, y_test, False), y_tr)


# Shape verification
for _ps, _ts, _nb in [(1, 1, 4), (6, 4, 10), (11, 4, 100)]:
    _gs  = math.ceil(_ps / _ts)
    _nt  = _gs * _gs
    _td  = _ts * _ts * _nb
    print('patch={:2d} token_size={} -> n_tokens={} token_dim={}'.format(
        _ps, _ts, _nt, _td))
print('Token geometry OK.')

In [ ]:
# ============================================================
# VISION TRANSFORMER MODEL (custom, from scratch in PyTorch)
#
# Uses nn.TransformerEncoderLayer (pure PyTorch, not HuggingFace/timm).
# pre-LN (norm_first=True) for more stable training.
# CLS token prepended; its output drives the classifier.
# ============================================================

class ViT(nn.Module):
    def __init__(self, n_tokens, token_dim,
                 embed_dim=64, n_heads=4, n_layers=4,
                 mlp_ratio=4, dropout=0.1, n_classes=2):
        super().__init__()

        # Linear projection: token_dim -> embed_dim
        self.proj = nn.Linear(token_dim, embed_dim)

        # Learnable CLS token and positional embeddings
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_tokens + 1, embed_dim))
        self.emb_drop  = nn.Dropout(dropout)

        # Transformer encoder (pre-LN, requires PyTorch >= 1.11)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=n_heads,
            dim_feedforward=embed_dim * mlp_ratio,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(embed_dim)

        # Classification head: CLS token -> n_classes
        self.head = nn.Linear(embed_dim, n_classes)

        # Weight initialisation
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.proj.weight, std=0.02)
        nn.init.trunc_normal_(self.head.weight, std=0.02)

    def forward(self, x):
        # x: (B, n_tokens, token_dim)
        B = x.shape[0]
        x = self.proj(x)                             # (B, n_tokens, embed_dim)

        cls = self.cls_token.expand(B, -1, -1)       # (B, 1, embed_dim)
        x   = torch.cat([cls, x], dim=1)             # (B, n_tokens+1, embed_dim)
        x   = self.emb_drop(x + self.pos_embed)

        x   = self.transformer(x)                    # (B, n_tokens+1, embed_dim)
        cls_out = self.norm(x[:, 0])                 # (B, embed_dim)
        return self.head(cls_out)                    # (B, n_classes)


# Quick shape test
for _ps, _ts, _nb in [(1, 1, 4), (6, 4, 10), (11, 4, 100)]:
    _gs = math.ceil(_ps / _ts)
    _nt = _gs * _gs
    _td = _ts * _ts * _nb
    _m  = ViT(_nt, _td, EMBED_DIM, N_HEADS, N_LAYERS, MLP_RATIO, DROPOUT).to(DEVICE)
    _x  = torch.randn(2, _nt, _td).to(DEVICE)
    _out = _m(_x)
    print('patch={:2d} ts={} n_tokens={:2d} token_dim={:4d} -> output {}'.format(
        _ps, _ts, _nt, _td, tuple(_out.shape)))
del _m, _x, _out
print('Model OK.')

In [ ]:
# ============================================================
# TRAINING UTILITIES
# ============================================================

class EarlyStopping:
    def __init__(self, patience=10):
        self.patience     = patience
        self.best_loss    = float('inf')
        self.counter      = 0
        self.best_weights = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss    = val_loss
            self.counter      = 0
            self.best_weights = {k: v.cpu().clone()
                                 for k, v in model.state_dict().items()}
        else:
            self.counter += 1
        return self.counter >= self.patience

    def restore_best(self, model):
        if self.best_weights is not None:
            model.load_state_dict(self.best_weights)


def make_criterion(y_train, loss_fn, device):
    """Build loss function with class weights."""
    weights = compute_class_weights(y_train)
    w = torch.tensor([weights[0], weights[1]], dtype=torch.float32).to(device)
    if loss_fn == 'CrossEntropy':
        return nn.CrossEntropyLoss(weight=w)
    else:  # BCE
        return nn.BCEWithLogitsLoss(pos_weight=w[1:2])


def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        logits = model(X_b)
        loss   = criterion(logits, y_b)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y_b)
    return total_loss / len(loader.dataset)


def evaluate_loader(model, loader, criterion, device):
    """Return (avg_loss, preds, probs, labels)."""
    model.eval()
    total_loss, preds, probs, labels = 0.0, [], [], []
    with torch.no_grad():
        for X_b, y_b in loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            logits = model(X_b)
            total_loss += criterion(logits, y_b).item() * len(y_b)
            if LOSS_FN == 'BCE':
                prob = torch.sigmoid(logits[:, 0])
                preds.extend((prob > 0.5).long().cpu().numpy())
            else:
                prob = torch.softmax(logits, dim=1)[:, 1]
                preds.extend(logits.argmax(1).cpu().numpy())
            probs.extend(prob.cpu().numpy())
            labels.extend(y_b.cpu().numpy())
    return (total_loss / len(loader.dataset),
            np.array(preds), np.array(probs), np.array(labels))


def train_fold(model, train_loader, val_loader, y_train, n_epochs,
               es_patience, lr_init, warmup_epochs,
               lr_factor, lr_patience, lr_min, device):
    """
    Full training loop with linear LR warmup + ReduceLROnPlateau.

    Warmup: epochs 0..(warmup_epochs-1) linearly scale lr from
            lr_init/warmup_epochs up to lr_init.
    After warmup: ReduceLROnPlateau on val_loss.

    Returns (train_losses, val_losses, n_epochs_run).
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr_init)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=lr_factor,
                                  patience=lr_patience, min_lr=lr_min)
    criterion = make_criterion(y_train, LOSS_FN, device)
    stopper   = EarlyStopping(patience=es_patience)
    train_losses, val_losses = [], []

    ep_bar = tqdm(range(n_epochs), desc='    Epochs', leave=False, unit='ep')
    for epoch in ep_bar:
        # Linear warmup: override lr each warmup epoch
        if epoch < warmup_epochs:
            lr_now = lr_init * (epoch + 1) / warmup_epochs
            for pg in optimizer.param_groups:
                pg['lr'] = lr_now

        tr_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        vl_loss, _, _, _ = evaluate_loader(model, val_loader, criterion, device)

        if epoch >= warmup_epochs:
            scheduler.step(vl_loss)

        train_losses.append(tr_loss)
        val_losses.append(vl_loss)
        ep_bar.set_postfix({'tr':  '{:.4f}'.format(tr_loss),
                            'val': '{:.4f}'.format(vl_loss),
                            'lr':  '{:.2e}'.format(optimizer.param_groups[0]['lr'])})
        if stopper.step(vl_loss, model):
            break

    stopper.restore_best(model)
    return train_losses, val_losses, len(train_losses)

print('Training utilities defined.')

In [ ]:
# ============================================================
# MAIN TRAINING LOOP
# 16 combos x 3 folds = 48 runs  (patch=11, token_size=4)
# ============================================================

folds = get_lopocv_folds()
grid  = get_experiment_grid()
total_runs = len(grid) * len(folds)
run_num    = 0

# Token geometry for main runs
_gs_main   = math.ceil(PATCH_SIZE / TOKEN_SIZE)
N_TOKENS_MAIN = _gs_main * _gs_main

print('ViT {}  |  {} combos x {} folds = {} runs'.format(
    VERSION, len(grid), len(folds), total_runs))
print('Patch={} token_size={} n_tokens={}'.format(
    PATCH_SIZE, TOKEN_SIZE, N_TOKENS_MAIN))
print('Results ->', RESULTS_CSV)
print()

combo_bar = tqdm(grid, desc='Combos', unit='combo')
for method, n_bands in combo_bar:
    band_indices = load_band_indices(method, n_bands)
    combo_bar.set_postfix({'method': method, 'bands': n_bands})

    fold_bar = tqdm(folds, desc='  Folds', unit='fold', leave=False)
    for fold in fold_bar:
        fold_num = fold['fold']
        run_num += 1
        fold_bar.set_postfix({'fold': fold_num,
                              'run': '{}/{}'.format(run_num, total_runs)})

        if is_done(RESULTS_CSV, MODEL, method, n_bands, PATCH_SIZE, TOKEN_SIZE, fold_num):
            fold_bar.write('  [skip] {}/{} fold={}'.format(method, n_bands, fold_num))
            continue

        # ---- Load data ----
        train_ldr, val_ldr, test_ldr, y_train = make_loaders(
            fold, band_indices, PATCH_SIZE, TOKEN_SIZE,
            PATCHES_PER_ROI, RANDOM_SEED, BATCH_SIZE)

        # ---- Token geometry for this combo ----
        n_tokens  = N_TOKENS_MAIN
        token_dim = TOKEN_SIZE * TOKEN_SIZE * n_bands

        # ---- Build model ----
        torch.manual_seed(RANDOM_SEED)
        model = ViT(n_tokens, token_dim, EMBED_DIM, N_HEADS, N_LAYERS,
                    MLP_RATIO, DROPOUT).to(DEVICE)

        # ---- Train ----
        t_train = time.time()
        train_losses, val_losses, n_epochs_run = train_fold(
            model, train_ldr, val_ldr, y_train,
            MAX_EPOCHS, ES_PATIENCE, LR_INIT, WARMUP_EPOCHS,
            LR_FACTOR, LR_PATIENCE, LR_MIN, DEVICE)
        train_time_sec = time.time() - t_train

        # ---- Evaluate on test ----
        criterion = make_criterion(y_train, LOSS_FN, DEVICE)
        t_inf = time.time()
        _, preds, probs, labels = evaluate_loader(model, test_ldr, criterion, DEVICE)
        n_test = len(labels)
        inf_ms = (time.time() - t_inf) / n_test * 1000

        row = {
            'model':                       MODEL,
            'method':                      method,
            'n_bands':                     n_bands,
            'patch_size':                  PATCH_SIZE,
            'token_size':                  TOKEN_SIZE,
            'fold':                        fold_num,
            'loss_fn':                     LOSS_FN,
            'accuracy':                    round(accuracy_score(labels, preds), 6),
            'sensitivity':                 round(recall_score(labels, preds, pos_label=1,
                                                              zero_division=0), 6),
            'specificity':                 round(recall_score(labels, preds, pos_label=0,
                                                              zero_division=0), 6),
            'f1':                          round(f1_score(labels, preds, average='macro',
                                                          zero_division=0), 6),
            'auc':                         round(roc_auc_score(labels, probs), 6),
            'train_time_sec':              round(train_time_sec, 3),
            'inference_time_per_image_ms': round(inf_ms, 6),
        }

        # ---- Save to Drive ----
        append_row(RESULTS_CSV, row, CSV_COLS)

        curve_name = 'curve_{}_{}b_p{}_t{}_f{}.png'.format(
            method, n_bands, PATCH_SIZE, TOKEN_SIZE, fold_num)
        save_learning_curve(
            train_losses, val_losses,
            '{} {}/{}b patch={} ts={} fold={} ({} ep)'.format(
                MODEL, method, n_bands, PATCH_SIZE, TOKEN_SIZE, fold_num, n_epochs_run),
            PLOTS_DIR / curve_name)

        fold_bar.write(
            '  [{:2d}/{:2d}] fold={} {}/{:3d}  '
            'acc={:.3f} sens={:.3f} spec={:.3f} f1={:.3f} auc={:.3f}  '
            'train={:.1f}s ep={}'.format(
                run_num, total_runs, fold_num, method, n_bands,
                row['accuracy'], row['sensitivity'], row['specificity'],
                row['f1'], row['auc'], train_time_sec, n_epochs_run))

        del model
        torch.cuda.empty_cache()

print()
print('All main runs complete.')

In [ ]:
# Generate summary CSV (only if results exist)
if RESULTS_CSV.exists():
    generate_summary(RESULTS_CSV, SUMMARY_CSV, CSV_COLS, METRIC_COLS)
    print('Main experiment done.')
else:
    print('No results yet — run the main training loop first.')

## Ablation: Patch Size Sweep

Runs each combo at patch sizes **1x1, 6x6, 11x11**.  
- patch=1: uses `token_size=1` (pixel-level ViT, 1 token = full spectrum)  
- patch=6: uses `token_size=4` (2x2 grid = 4 tokens)  
- patch=11: uses `token_size=4` (3x3 grid = 9 tokens)  

Results saved to `vit_v1_ablation.csv` (separate from main results).  
Main runs (patch=11, token_size=4) are re-included with `is_ablation=True` so the ablation CSV is self-contained.

In [ ]:
# ============================================================
# ABLATION LOOP: patch sizes 1, 6, 11
# token_size=1 for patch=1 (pixel-level), TOKEN_SIZE for others
# ============================================================

abl_total = len(ABLATION_SIZES) * len(grid) * len(folds)
abl_run   = 0

print('Ablation: {} patch sizes x {} combos x {} folds = {} runs'.format(
    len(ABLATION_SIZES), len(grid), len(folds), abl_total))
print('Ablation CSV ->', ABLATION_CSV)
print()

size_bar = tqdm(ABLATION_SIZES, desc='Patch sizes', unit='size')
for patch_size in size_bar:
    # patch=1 -> pixel-level ViT (token_size=1 avoids reflect-padding a single pixel)
    abl_token_size = 1 if patch_size == 1 else TOKEN_SIZE
    size_bar.set_postfix({'patch': '{}x{}'.format(patch_size, patch_size),
                          'ts': abl_token_size})

    # Token geometry for this patch/token_size combo
    abl_gs        = math.ceil(patch_size / abl_token_size)
    abl_n_tokens  = abl_gs * abl_gs

    combo_bar = tqdm(grid, desc='  Combos', unit='combo', leave=False)
    for method, n_bands in combo_bar:
        band_indices = load_band_indices(method, n_bands)
        combo_bar.set_postfix({'method': method, 'bands': n_bands})
        abl_token_dim = abl_token_size * abl_token_size * n_bands

        fold_bar = tqdm(folds, desc='    Folds', unit='fold', leave=False)
        for fold in fold_bar:
            fold_num = fold['fold']
            abl_run += 1
            fold_bar.set_postfix({'fold': fold_num,
                                  'run': '{}/{}'.format(abl_run, abl_total)})

            if is_done(ABLATION_CSV, MODEL, method, n_bands,
                       patch_size, abl_token_size, fold_num):
                fold_bar.write('  [skip] patch={} ts={} {}/{} fold={}'.format(
                    patch_size, abl_token_size, method, n_bands, fold_num))
                continue

            abl_seed = RANDOM_SEED + patch_size * 100

            train_ldr, val_ldr, test_ldr, y_train = make_loaders(
                fold, band_indices, patch_size, abl_token_size,
                PATCHES_PER_ROI, abl_seed, BATCH_SIZE)

            torch.manual_seed(RANDOM_SEED)
            model = ViT(abl_n_tokens, abl_token_dim, EMBED_DIM, N_HEADS, N_LAYERS,
                        MLP_RATIO, DROPOUT).to(DEVICE)

            t_train = time.time()
            train_losses, val_losses, n_ep = train_fold(
                model, train_ldr, val_ldr, y_train,
                MAX_EPOCHS, ES_PATIENCE, LR_INIT, WARMUP_EPOCHS,
                LR_FACTOR, LR_PATIENCE, LR_MIN, DEVICE)
            train_time_sec = time.time() - t_train

            criterion = make_criterion(y_train, LOSS_FN, DEVICE)
            t_inf = time.time()
            _, preds, probs, labels = evaluate_loader(model, test_ldr, criterion, DEVICE)
            inf_ms = (time.time() - t_inf) / len(labels) * 1000

            row = {
                'model':                       MODEL,
                'method':                      method,
                'n_bands':                     n_bands,
                'patch_size':                  patch_size,
                'token_size':                  abl_token_size,
                'fold':                        fold_num,
                'loss_fn':                     LOSS_FN,
                'accuracy':                    round(accuracy_score(labels, preds), 6),
                'sensitivity':                 round(recall_score(labels, preds, pos_label=1,
                                                                  zero_division=0), 6),
                'specificity':                 round(recall_score(labels, preds, pos_label=0,
                                                                  zero_division=0), 6),
                'f1':                          round(f1_score(labels, preds, average='macro',
                                                              zero_division=0), 6),
                'auc':                         round(roc_auc_score(labels, probs), 6),
                'train_time_sec':              round(train_time_sec, 3),
                'inference_time_per_image_ms': round(inf_ms, 6),
                'is_ablation':                 True,
            }
            append_row(ABLATION_CSV, row, ABLATION_COLS)

            curve_name = 'ablation_{}_{}_{}b_p{}_t{}_f{}.png'.format(
                VERSION, method, n_bands, patch_size, abl_token_size, fold_num)
            save_learning_curve(
                train_losses, val_losses,
                'Ablation patch={} ts={} {}/{}b fold={} ({} ep)'.format(
                    patch_size, abl_token_size, method, n_bands, fold_num, n_ep),
                PLOTS_DIR / curve_name)

            fold_bar.write(
                '  [{:3d}/{:3d}] patch={} ts={} {}/{:3d} fold={}  '
                'f1={:.3f} auc={:.3f} ep={}'.format(
                    abl_run, abl_total, patch_size, abl_token_size,
                    method, n_bands, fold_num,
                    row['f1'], row['auc'], n_ep))

            del model
            torch.cuda.empty_cache()

print()
print('Ablation complete.')
generate_summary(ABLATION_CSV,
                 RESULTS_DIR / 'vit_{}_ablation_summary.csv'.format(VERSION),
                 ABLATION_COLS, METRIC_COLS)